In [1]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

In [3]:
df_coord_numbs = pd.read_csv("data_cod/coord_numbs.csv")
df_coord_numbs.rename(columns={"smiles": "can_smiles"}, inplace=True)

In [4]:
# Drop rows and columns that contain only zeros:

rows_only_zeros = df_coord_numbs[(df_coord_numbs == 0).all(axis=1)]
columns_only_zeros = df_coord_numbs.loc[:, (df_coord_numbs == 0).all(axis=0)]

df_coord_numbs = df_coord_numbs.drop(columns=columns_only_zeros.columns, index=rows_only_zeros.index)

In [5]:
df_merged_temp = pd.read_csv("data_cod/cod_bradley_merged.csv")
df_merged_temp = df_merged_temp[df_merged_temp["id"].isin(df_coord_numbs["id"])]

# Only bradley, to use part of it for test:
df_bradley_temp = pd.read_csv("data_cod/bradley_with_cif.csv")
df_bradley_temp = df_bradley_temp[df_bradley_temp["id"].isin(df_coord_numbs["id"])]

---
## Creating Train and Test splits:
- 0.2 of Bradley is test, rest is train

In [6]:
TEST_BRADLEY_FRAC = 0.2

In [7]:
df_test = df_bradley_temp.sample(frac=TEST_BRADLEY_FRAC, random_state=42)

df_train = df_merged_temp.drop(index=df_merged_temp[df_merged_temp["id"].isin(df_test["id"])].index)

In [8]:
df_train = pd.merge(df_train, df_coord_numbs, on=["id", "can_smiles"], how="inner")
df_test = pd.merge(df_test, df_coord_numbs, on=["id", "can_smiles"], how="inner")

In [9]:
df_train

,id,can_smiles,T,cif_path,Ag–Ag,Al–H,H–Al,As–As,B–B,B–H,...,Se–Se,Si–H,H–Si,Si–Si,Sn–H,H–Sn,Sr–Sr,Ti–Ti,Zn–H,H–Zn
0,1000018,CC1(C)[C@@H]2CC[C@]3(C2)[C@H](O)CC[C@@H](O)[C@...,180.0,cifs/1000018.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1000019,CN(C)c1ccc(/C=C/C(=O)c2ccccc2O)cc1,-97.0,cifs/1000019.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1008189,OO,-40.0,cifs/1008189.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1008775,NC(N)=O,134.0,cifs/1008775.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1008776,NC(N)=O,134.0,cifs/1008776.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11852,9014479,ClCl,-101.0,cifs/9014479.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11853,9015081,c1ccc2c(c1)Cc1ccccc1-2,116.0,cifs/9015081.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11854,9015108,Cl,-114.2,cifs/9015108.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11855,9015583,ClCl,-101.0,cifs/9015583.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
df_train

,id,can_smiles,T,cif_path,Ag–Ag,Al–H,H–Al,As–As,B–B,B–H,...,Se–Se,Si–H,H–Si,Si–Si,Sn–H,H–Sn,Sr–Sr,Ti–Ti,Zn–H,H–Zn
0,1000018,CC1(C)[C@@H]2CC[C@]3(C2)[C@H](O)CC[C@@H](O)[C@...,180.0,cifs/1000018.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1000019,CN(C)c1ccc(/C=C/C(=O)c2ccccc2O)cc1,-97.0,cifs/1000019.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1008189,OO,-40.0,cifs/1008189.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1008775,NC(N)=O,134.0,cifs/1008775.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1008776,NC(N)=O,134.0,cifs/1008776.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11852,9014479,ClCl,-101.0,cifs/9014479.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11853,9015081,c1ccc2c(c1)Cc1ccccc1-2,116.0,cifs/9015081.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11854,9015108,Cl,-114.2,cifs/9015108.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11855,9015583,ClCl,-101.0,cifs/9015583.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
X_train = df_train.drop(columns=["id", "can_smiles", "T", "cif_path"])
y_train = df_train["T"].astype(float)

X_test = df_test.drop(columns=["id", "can_smiles", "T"])
y_test = df_test["T"].astype(float)

---
## Training Catboost:

In [12]:
import optuna
from catboost import CatBoostRegressor
from src.utils import eval_metrics

from sklearn.decomposition import PCA

import numpy as np


In [ ]:
# Objective function
def objective(trial):
    n_components = trial.suggest_int("pca_components", 2, X_train.shape[1])

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    params = {
        "iterations": trial.suggest_int("iterations", 200, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 14),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 5.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "verbose": 0,
        "task_type": "CPU"
    }

    model = CatBoostRegressor(**params)
    model.fit(X_train_pca, y_train, eval_set=(X_test_pca, y_test), early_stopping_rounds=30)

    y_pred = model.predict(X_test_pca)
    return eval_metrics(y_test, y_pred, "regression")["R2"]

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000, n_jobs=32)

# Train final model with best params
best_params = study.best_params
best_params["loss_function"] = "RMSE"
best_params["verbose"] = 0

[I 2025-04-24 12:20:07,030] A new study created in memory with name: no-name-a447c853-7d6a-44f0-9206-6d63c6e95a1f


[I 2025-04-24 12:20:15,342] Trial 20 finished with value: 0.562616315857208 and parameters: {'pca_components': 44, 'iterations': 464, 'learning_rate': 0.1366570792851537, 'depth': 8, 'l2_leaf_reg': 0.10973588767995286, 'bagging_temperature': 0.33568844343585136, 'random_strength': 0.010407508933925108, 'border_count': 95}. Best is trial 20 with value: 0.562616315857208.
[I 2025-04-24 12:20:16,082] Trial 15 finished with value: 0.20266664827245928 and parameters: {'pca_components': 29, 'iterations': 332, 'learning_rate': 0.0028793059846204372, 'depth': 5, 'l2_leaf_reg': 0.023443525957596353, 'bagging_temperature': 0.35376842460149216, 'random_strength': 0.09406771195667414, 'border_count': 152}. Best is trial 20 with value: 0.562616315857208.
[I 2025-04-24 12:20:16,296] Trial 11 finished with value: 0.20869820951634188 and parameters: {'pca_components': 10, 'iterations': 293, 'learning_rate': 0.0025517322530007078, 'depth': 7, 'l2_leaf_reg': 1.6182082922891003, 'bagging_temperature': 0.

In [ ]:
final_model = CatBoostRegressor(**best_params)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print(eval_metrics(y_test, y_pred))
